# RAG desde cero (versión gratis y local)
### Retrieval-Augmented Generation sin APIs de pago

**Universidad de Santiago de Chile (USACH)**  
Diplomado de Inteligencia artificial aplicada a la industria tecnológica  
Reelaborado por Diego Fischer

*Basado en la serie "RAG From Scratch" de LangChain (Lance Martin). Esta versión migra todo el pipeline a un stack **100% abierto y gratuito** — sin OpenAI, sin claves de API, sin costos — para que corra completo en Colab.*

---

En este cuaderno construimos, pieza por pieza, un sistema **RAG** (*Retrieval-Augmented Generation*): un LLM que responde preguntas apoyándose en **tus propios documentos**, en vez de solo en lo que memorizó durante su entrenamiento.

| Componente | Versión original (de pago) | Esta versión (gratis/local) |
|---|---|---|
| Embeddings | `OpenAIEmbeddings` (API) | `HuggingFaceEmbeddings` (local) |
| Base vectorial | Chroma | Chroma (ya era local) |
| LLM generador | `gpt-3.5-turbo` (API) | `Qwen2.5` abierto (local) |
| Costo / clave | requiere tarjeta + API key | **cero costo, sin claves** |

> **¿Por qué migrar a local?** Tres razones que vale la pena discutir con los estudiantes: (1) **costo** — nadie necesita pagar ni registrar tarjeta; (2) **privacidad** — los documentos nunca salen del entorno, algo crítico con datos sensibles; (3) **reproducibilidad** — el cuaderno no depende de que una API externa siga viva o mantenga precios.

---

## ¿Qué es RAG y por qué lo necesitamos?

Un LLM (como GPT, Llama o Mistral) tiene dos limitaciones incómodas:

1. **Su conocimiento está congelado** en la fecha de su entrenamiento. No sabe nada de tus documentos internos, de lo que pasó ayer, ni de tu empresa o tu curso.
2. **Alucina con seguridad.** Cuando no sabe algo, muchas veces se lo inventa con total confianza, lo cual es peor que decir "no sé".

**RAG** resuelve ambas cosas con una idea simple y poderosa: en vez de pedirle al LLM que responda de memoria, primero **buscamos** los fragmentos relevantes en una colección de documentos nuestros, y se los **entregamos como contexto** junto con la pregunta. El LLM ya no responde de memoria: responde leyendo.

La analogía perfecta: es la diferencia entre un **examen a libro cerrado** y uno a **libro abierto**. RAG convierte a un LLM "a libro cerrado" (que responde de memoria y a veces inventa) en uno "a libro abierto" (que consulta la fuente antes de responder). Y como el LLM responde desde un texto que le dimos, podemos rastrear de dónde salió cada respuesta.

### Las tres etapas de RAG

Todo sistema RAG tiene la misma anatomía, y este cuaderno sigue exactamente ese orden:

```
   [ INDEXAR ]  ->  [ RECUPERAR ]  ->  [ GENERAR ]
   preparar los     buscar los         responder usando
   documentos       fragmentos         esos fragmentos
   (una vez)        relevantes         como contexto
```

1. **Indexar** (*Indexing*): partimos los documentos en fragmentos, los convertimos en vectores (*embeddings*) y los guardamos en una base vectorial. Esto se hace una sola vez.
2. **Recuperar** (*Retrieval*): dada una pregunta, buscamos en la base los fragmentos más parecidos semánticamente.
3. **Generar** (*Generation*): le pasamos al LLM la pregunta + esos fragmentos, y produce la respuesta apoyada en ellos.

---

## Fase 1 — Instalación y carga de modelos

Instalamos las piezas de LangChain más los modelos abiertos. Nota lo que **no** aparece: ninguna clave de API, ningún `os.environ['OPENAI_API_KEY']`. Esa es justamente la gracia de esta versión.

> **Activa la GPU** (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*): el LLM generador corre mucho más rápido, aunque también funciona en CPU.

In [1]:
!pip install -q -U langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers chromadb beautifulsoup4 transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/1

### Cargar el modelo de embeddings

Un *embedding* convierte un texto en un vector de números que captura su **significado**: textos con sentido parecido quedan cerca en el espacio vectorial. Es el ingrediente que permite "buscar por significado" y no solo por palabras exactas.

Elegimos un modelo **multilingüe** y pequeño (`paraphrase-multilingual-MiniLM-L12-v2`), que entiende español e inglés y corre rápido incluso en CPU. La primera vez se descarga (unos segundos).

In [2]:
import os
os.environ['USER_AGENT'] = 'usach-rag-curso'  # evita un warning de WebBaseLoader

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

# Prueba rápida: ¿de qué tamaño es un vector de embedding?
vec = embeddings.embed_query('Hola mundo')
print(f'Un texto se convierte en un vector de {len(vec)} dimensiones.')
print('Primeros 5 valores:', [round(x, 4) for x in vec[:5]])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Un texto se convierte en un vector de 384 dimensiones.
Primeros 5 valores: [0.1216, 0.2608, 0.1357, 0.1036, -0.0832]


### Cargar el LLM generador (local)

Como generador usamos `Qwen/Qwen2.5-1.5B-Instruct`: un modelo abierto (licencia Apache-2.0, sin restricciones de acceso), pequeño, que sigue instrucciones, maneja español y cabe cómodo en la GPU gratuita de Colab (también corre en CPU, más lento). Lo envolvemos con `HuggingFacePipeline` para que LangChain lo trate igual que trataría a un modelo de OpenAI.

Usamos la tarea `text-generation` (modelos *decoder-only*). Un detalle de versiones: las versiones recientes de `transformers` (5.x) eliminaron la antigua tarea `text2text-generation` que usaban modelos como flan-t5, así que un modelo de este tipo es hoy la opción robusta.

> **Para mejores respuestas** puedes subir a `Qwen/Qwen2.5-3B-Instruct` o `Qwen/Qwen2.5-7B-Instruct` (este último con cuantización de 4 bits) o `microsoft/Phi-3-mini-4k-instruct`. Para aprender el flujo, la versión de 1.5B es rápida y suficiente.

In [3]:
import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print('Usando', 'GPU' if device == 0 else 'CPU')

# Modelo de chat abierto (decoder-only) -> tarea 'text-generation'
gen_pipe = pipeline(
    'text-generation',
    model='Qwen/Qwen2.5-1.5B-Instruct',
    device=device,
    torch_dtype=dtype,
    max_new_tokens=256,
    do_sample=False,          # respuestas deterministas
    return_full_text=False,   # devuelve solo la respuesta, no el prompt
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# Prueba: el LLM aún NO sabe de nuestros documentos, responde de memoria
print(llm.invoke('En una frase, ¿qué es la inteligencia artificial?'))

Usando GPU


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress

 La inteligencia artificial es el campo de las ciencias informáticas que se ocupa de crear sistemas y programas capaces de realizar tareas que normalmente requieren un cerebro humano. 

Esta definición puede variar ligeramente dependiendo del contexto o de quién la use, pero en general se refiere a la capacidad de un sistema para aprender, razonar, tomar decisiones, entender el lenguaje natural, etc., utilizando datos y algoritmos.

La IA no solo busca replicar los procesos mentales humanos, sino también mejorarlos, adaptarse a nuevas situaciones y superar limitaciones inherentes a la mente humana. Esto incluye desde simples tareas como reconocimiento facial hasta complejos problemas de planificación y resolución de conflictos. En última instancia, la IA tiene como objetivo final ser más eficiente y efectiva que un ser humano en ciertas áreas específicas. 

Es importante notar que aunque la IA ha avanzado significativamente en muchos aspectos, todavía está lejos de alcanzar la totalida

## Fase 2 — Indexado (Indexing)

Vamos a construir la base de conocimiento. Usaremos como documento de ejemplo un artículo de blog sobre agentes basados en LLM (de Lilian Weng). El flujo de indexado tiene cuatro pasos: **cargar → trocear → embeber → guardar**.

### Un primer experimento: embeddings y similitud

Antes de indexar el blog, entendamos el corazón de RAG con un ejemplo mínimo. Convertimos dos frases en vectores y medimos cuán "cercanas" están con la **similitud del coseno** (1 = idénticas en significado, 0 = sin relación). Esta es exactamente la operación que, a gran escala, permite "buscar por significado".

In [4]:
import numpy as np

def similitud_coseno(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

pregunta  = '¿Qué tipo de mascotas me gustan?'
doc_bueno = 'Mi mascota favorita es un gato.'
doc_malo  = 'El PIB de Chile creció el año pasado.'

v_preg = embeddings.embed_query(pregunta)
v_bueno = embeddings.embed_query(doc_bueno)
v_malo = embeddings.embed_query(doc_malo)

print(f'Similitud con frase relacionada : {similitud_coseno(v_preg, v_bueno):.4f}')
print(f'Similitud con frase sin relación: {similitud_coseno(v_preg, v_malo):.4f}')

Similitud con frase relacionada : 0.6870
Similitud con frase sin relación: 0.0382


Fíjate cómo la frase sobre mascotas puntúa **mucho más alto** que la del PIB, aunque la pregunta no comparte casi ninguna palabra con ella ("mascotas" vs "gato"). Eso es búsqueda **semántica**: se basa en el significado, no en coincidir palabras. Es lo que hace a RAG superior a un simple *Ctrl+F*.

### Paso 1 — Cargar el documento

`WebBaseLoader` descarga una página web y extrae su texto. Con `bs4.SoupStrainer` le decimos que se quede solo con el contenido del artículo (título y cuerpo), descartando menús y pies de página.

In [5]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=('https://lilianweng.github.io/posts/2023-06-23-agent/',),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=('post-content', 'post-title', 'post-header')
        )
    ),
)
docs = loader.load()

print(f'Documentos cargados: {len(docs)}')
print(f'Caracteres totales : {len(docs[0].page_content)}')
print('\nPrimeros 300 caracteres:\n')
print(docs[0].page_content[:300])

/tmp/ipykernel_412/3416837102.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


Documentos cargados: 1
Caracteres totales : 43047

Primeros 300 caracteres:



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as i


### Paso 2 — Trocear (chunking) y por qué

No podemos meter el documento entero al LLM: los modelos tienen una **ventana de contexto** limitada (un máximo de *tokens* que pueden leer de una vez). Además, si le diéramos todo, el fragmento realmente relevante se "diluiría" entre páginas de texto irrelevante.

Por eso **troceamos** el documento en fragmentos (*chunks*). Dos parámetros clave:

- **`chunk_size`**: el tamaño de cada fragmento. Muy grande → poco preciso al recuperar; muy chico → se pierde el contexto. Es un equilibrio.
- **`chunk_overlap`**: cuánto se solapan fragmentos consecutivos. El solapamiento evita cortar una idea justo por la mitad entre dos fragmentos.

Usamos `RecursiveCharacterTextSplitter`, que intenta cortar respetando la estructura del texto (primero por párrafos, luego oraciones, luego palabras) para mantener juntas las ideas relacionadas.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # ~1000 caracteres por fragmento
    chunk_overlap=200,   # 200 caracteres de solapamiento entre fragmentos
)
splits = text_splitter.split_documents(docs)

print(f'El documento se dividió en {len(splits)} fragmentos.')
print('\nEjemplo de un fragmento:\n')
print(splits[10].page_content)

El documento se dividió en 63 fragmentos.

Ejemplo de un fragmento:

The paper hypothesizes that any algorithm that generates a set of learning histories can be distilled into a neural network by performing behavioral cloning over actions. The history data is generated by a set of source policies, each trained for a specific task. At the training stage, during each RL run, a random task is sampled and a subsequence of multi-episode history is used for training, such that the learned policy is task-agnostic.
In reality, the model has limited context window length, so episodes should be short enough to construct multi-episode history. Multi-episodic contexts of 2-4 episodes are necessary to learn a near-optimal in-context RL algorithm. The emergence of in-context RL requires long enough context.


> **¿Cuántos tokens es eso?** Los modelos no cuentan caracteres sino *tokens* (trozos de palabra, ~4 caracteres cada uno en promedio). Veamos cuántos tokens ve nuestro LLM en un fragmento — esto importa para no exceder su ventana de contexto.

In [7]:
# Contamos tokens con el tokenizador del propio LLM
tok = gen_pipe.tokenizer
n_tokens = len(tok(splits[10].page_content)['input_ids'])
n_chars = len(splits[10].page_content)
print(f'Fragmento: {n_chars} caracteres  ->  {n_tokens} tokens')
print(f'Razón aproximada: {n_chars / n_tokens:.1f} caracteres por token')

Fragmento: 735 caracteres  ->  147 tokens
Razón aproximada: 5.0 caracteres por token


### Paso 3 y 4 — Embeber y guardar en la base vectorial

Ahora convertimos cada fragmento en su vector (embedding) y los almacenamos en **Chroma**, una base de datos vectorial que corre localmente. Chroma se encarga de, dado un vector de consulta, encontrar rápidamente los fragmentos más cercanos.

`Chroma.from_documents` hace las dos cosas de una vez: embebe todos los fragmentos (usando nuestro modelo local) y los indexa.

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
)

print(f'Base vectorial creada con {len(splits)} fragmentos indexados.')

Base vectorial creada con 63 fragmentos indexados.


## Fase 3 — Recuperación (Retrieval)

Con la base lista, el **retriever** es el objeto que, dada una pregunta, devuelve los fragmentos más relevantes. Por dentro: embebe la pregunta con el mismo modelo, y busca los fragmentos cuyo vector está más cerca (mayor similitud de coseno).

El parámetro `k` controla **cuántos** fragmentos recuperar. Pocos → riesgo de quedarnos cortos; muchos → metemos ruido y gastamos ventana de contexto.

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

pregunta = 'What is Task Decomposition?'
fragmentos = retriever.invoke(pregunta)

print(f'Se recuperaron {len(fragmentos)} fragmentos para la pregunta:')
print(f'  "{pregunta}"\n')
for i, frag in enumerate(fragmentos, 1):
    print(f'--- Fragmento {i} ---')
    print(frag.page_content[:250], '...\n')

Se recuperaron 3 fragmentos para la pregunta:
  "What is Task Decomposition?"

--- Fragmento 1 ---
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human input ...

--- Fragmento 2 ---
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model pe ...

--- Fragmento 3 ---
(3) Task execution: Expert models execute on the specific tasks and log results.
Instruction:

With the input and the inference results, the AI assistant needs to describe the process and results. The previous stages can be formed as - User Input: {{ ...



> **Nota:** la pregunta va en inglés porque el documento de ejemplo está en inglés (el modelo de embeddings es multilingüe, así que también entendería una pregunta en español y encontraría los fragmentos correctos — pruébalo). Para un RAG completamente en español, cargarías documentos en español y usarías un LLM generador con buen manejo del idioma.

## Fase 4 — Generación (Generation)

Último paso: entregar al LLM la pregunta **junto con** los fragmentos recuperados, y pedirle que responda **basándose solo en ese contexto**. Esa instrucción es crucial: es lo que evita que el modelo invente ("responde solo con el contexto; si no está, di que no lo sabes").

En LangChain esto se arma con un **prompt** (la plantilla de instrucción) que tiene huecos para `{context}` y `{question}`.

In [10]:
from langchain_core.prompts import PromptTemplate

plantilla = """Responde la pregunta usando ÚNICAMENTE el siguiente contexto.
Si la respuesta no aparece en el contexto, di que no tienes suficiente información.

Contexto:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate.from_template(plantilla)

# Veamos cómo queda el prompt ya rellenado (esto es lo que 'lee' el LLM)
contexto_texto = '\n\n'.join(f.page_content for f in fragmentos)
print(prompt.format(context=contexto_texto, question=pregunta)[:800], '...')

Responde la pregunta usando ÚNICAMENTE el siguiente contexto.
Si la respuesta no aparece en el contexto, di que no tienes suficiente información.

Contexto:
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a ...


In [11]:
# Generamos la respuesta pasando el prompt relleno al LLM
respuesta = llm.invoke(prompt.format(context=contexto_texto, question=pregunta))
print('RESPUESTA DEL MODELO:\n')
print(respuesta)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPUESTA DEL MODELO:

 Task Decomposition refers to breaking down a complex task into smaller, more manageable steps that an agent can understand and plan ahead. This process helps enhance model performance on complex tasks by utilizing the Chain of Thought (CoT) technique, which transforms big tasks into multiple manageable tasks through multiple-step reasoning. Tree of Thoughts further expands upon CoT by generating multiple thoughts per step, creating a tree structure for exploration. This method allows for a detailed understanding of the model's thinking process during task execution. The overall goal is to make complex tasks more accessible and easier to manage for agents. No information was provided about the context of the question regarding the AI assistant's response. 

No tengo suficiente información para responder esta pregunta de manera precisa y detallada basándome solo en el contexto proporcionado. La pregunta se refiere específicamente a una técnica llamada Task Decompo

> **Compara mentalmente:** si le hiciéramos esta misma pregunta al LLM **sin** el contexto (como en la prueba de la Fase 1), respondería de memoria y podría inventar. Con el contexto recuperado, la respuesta queda anclada al documento. Esa es, en una frase, toda la magia de RAG.

## Fase 5 — La cadena RAG completa

Hasta aquí ejecutamos los pasos a mano para entenderlos. Ahora los encadenamos en un solo objeto usando **LCEL** (*LangChain Expression Language*), la notación con `|` ("pipe") que conecta componentes como una tubería:

```
   pregunta -> [recuperar contexto] -> [armar prompt] -> [LLM] -> [texto final]
```

Cada `|` pasa la salida de un paso como entrada del siguiente. Esta única cadena hace todo el flujo RAG de punta a punta.

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def formatear_docs(docs):
    """Une los fragmentos recuperados en un solo bloque de texto."""
    return '\n\n'.join(doc.page_content for doc in docs)

# La cadena RAG completa
rag_chain = (
    {'context': retriever | formatear_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Ahora responder es UNA sola línea:
print(rag_chain.invoke('What is Task Decomposition?'))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Task Decomposition refers to breaking down a complex task into smaller, more manageable steps that an agent can understand and plan ahead. This process helps enhance model performance on complex tasks by utilizing the Chain of Thought (CoT) technique, which transforms big tasks into multiple manageable tasks through multiple-step reasoning. Tree of Thoughts further expands upon CoT by generating multiple thoughts per step, creating a tree structure for exploration. This method allows for a detailed understanding of the model's thinking process during task execution. The overall goal is to make complex tasks more accessible and easier to manage for agents. No information was provided about the context of the question regarding the AI assistant's response. 

No tengo suficiente información para responder esta pregunta de manera precisa y detallada basándome solo en el contexto proporcionado. La pregunta se refiere específicamente a una técnica llamada Task Decomposition, pero no hay nin

### Pruébalo con otras preguntas

La cadena ya está lista: cualquier pregunta sobre el documento indexado se responde llamando `rag_chain.invoke(...)`. El *retriever* busca los fragmentos pertinentes y el LLM responde con ellos.

In [13]:
preguntas = [
    'What are the types of memory in an LLM agent?',
    'What is self-reflection?',
]

for p in preguntas:
    print(f'PREGUNTA: {p}')
    print(f'RESPUESTA: {rag_chain.invoke(p)}\n')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PREGUNTA: What are the types of memory in an LLM agent?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPUESTA:  In an LLM agent, there are three main types of memory:

1. Short-term memory: Used for processing immediate inputs and outputs.
2. Long-term memory: Stores vast amounts of information over extended periods.
3. External memory: Utilized through API calls for additional information not present in the model's weights.

La respuesta es correcta. La memoria del modelo se divide en tres tipos principales: memoria corta, memoria larga y memoria externa. La memoria corta se utiliza para procesar entradas y salidas inmediatas, mientras que la memoria larga se encarga de almacenar grandes cantidades de información durante períodos prolongados. Finalmente, la memoria externa se utiliza mediante llamadas a APIs para obtener información adicional que no está presente en los pesos del modelo. No tengo suficiente información para responder sobre las diferencias entre AD, ED, source policy y RL^2 o sobre cómo estos componentes interactúan con el sistema de memoria.

PREGUNTA: What is self-

---
## Licencia y créditos

[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/deed.es)

Este cuaderno está licenciado bajo una [Licencia Creative Commons Atribución 4.0 Internacional (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/deed.es). Puede compartirse y adaptarse, incluso con fines comerciales, siempre que se reconozca la autoría de forma adecuada.

**Reelaboración, migración a stack local y explicaciones:** Diego Fischer — Universidad de Santiago de Chile (USACH).  
**Material original:** serie *"RAG From Scratch"* de LangChain (Lance Martin).  
**Documento de ejemplo:** *"LLM Powered Autonomous Agents"* de Lilian Weng.  
**Modelos:** `paraphrase-multilingual-MiniLM-L12-v2` y `Qwen/Qwen2.5-1.5B-Instruct` vía HuggingFace.